# Task plan 생성

In [24]:
# Initialize env, agent
from concept.agent import Agent
from concept.env import Env
from test import load_tasks_and_constraints
from task_management.planner.exhaustive_planner import ExhaustivePlanner
from utils.util import get_paths_to_leaves


env = Env()
env.gen_dummy()
agent = Agent("Waiting", "Living Room", env)

# Task Plan generation
task_name, tasks, constraints = load_tasks_and_constraints()
task_plans, opt_task_plans = ExhaustivePlanner(
    agent, tasks, constraints
).generate_valid_plans()
opt_task_plans = get_paths_to_leaves(opt_task_plans)

Number of ordered paths: 1/1
Makespan: 9.75
Optimal Paths : ['Start', 'Move (Living Room -> Kitchen:Toaster)', 'Place Bread in Toaster', 'Turn on Toaster', 'Wait_for_Turn off Toaster', 'Turn off Toaster', 'Remove Toast from Toaster']



# Simulation, Task Trajectory 생성

## 1. Initialization
- Task plan을 Simulation에서 수행한 결과를 얻음
- Simulation이란, 실제 Task 수행 결과를 얻고자 위함이었음

In [25]:
import numpy as np
from collections import namedtuple

SimTask = namedtuple("SimTask", ["name", "start", "end", "duration"])
PlanTask = namedtuple("PlanTask", ["name", "start", "end", "duration"])

In [26]:
# 여러 번의 시뮬레이션
np.random.seed(42)  # 재현성을 위한 시드 설정
plan_tasks = []
sim_tasks = []

sim_task_start_time = 0
sim_task_start_time_temp = 0

# 직접 Task 1개에 대한 Bayesian Estimation 실행.
opt_task_plan = opt_task_plans[0]
for idx, opt_task in enumerate(opt_task_plan):
    if idx == 0:
        continue
    # Sample Task; Temporal Info
    opt_task_name = opt_task.name
    plan_task_duration = opt_task.duration
    plan_task_start_time = opt_task.makespan - opt_task.duration
    plan_task_end_time = opt_task.makespan

    plan_tasks.append(
        PlanTask(
            opt_task_name, plan_task_start_time, plan_task_end_time, plan_task_duration
        )
    )

    # Task Execution Info (Prior Knowledge)
    # Simulation에서, plan과 오차를 만들기 위함
    mu_prior, sigma_prior = plan_task_duration, 1

    # Task Real Info (Simulation)
    sim_task_start_time = round(sim_task_start_time_temp, 3)
    sim_task_duration = round(np.random.normal(mu_prior, sigma_prior), 3)
    sim_task_end_time = sim_task_start_time + sim_task_duration

    # 이전 Task가 끝난 시간은 다음 task의 시작 시간
    sim_task_start_time_temp = sim_task_end_time

    sim_tasks.append(
        SimTask(
            opt_task_name, sim_task_start_time, sim_task_end_time, sim_task_duration
        )
    )

In [27]:
for plan_task, sim_task in zip(plan_tasks, sim_tasks):
    print("Task Name :", plan_task.name)
    print("Plan info :", plan_task)
    print("Sim info", sim_task)
    print("Error", round(abs(plan_task.end - sim_task.end), 4))
    print()

Task Name : Move (Living Room -> Kitchen:Toaster)
Plan info : PlanTask(name='Move (Living Room -> Kitchen:Toaster)', start=0.0, end=0.75, duration=0.75)
Sim info SimTask(name='Move (Living Room -> Kitchen:Toaster)', start=0, end=1.247, duration=1.247)
Error 0.497

Task Name : Place Bread in Toaster
Plan info : PlanTask(name='Place Bread in Toaster', start=0.75, end=1.75, duration=1)
Sim info SimTask(name='Place Bread in Toaster', start=1.247, end=2.109, duration=0.862)
Error 0.359

Task Name : Turn on Toaster
Plan info : PlanTask(name='Turn on Toaster', start=1.75, end=2.75, duration=1)
Sim info SimTask(name='Turn on Toaster', start=2.109, end=3.7569999999999997, duration=1.648)
Error 1.007

Task Name : Wait_for_Turn off Toaster
Plan info : PlanTask(name='Wait_for_Turn off Toaster', start=2.75, end=7.75, duration=5.0)
Sim info SimTask(name='Wait_for_Turn off Toaster', start=3.757, end=10.28, duration=6.523)
Error 2.53

Task Name : Turn off Toaster
Plan info : PlanTask(name='Turn off To

# Execution
## 1. Assumption
- Plan의 Task Duration은 prior knowledge
- Initial Distribution은 $n(\hat{t_e}, (\hat{t_e}-t_c))$

## 2. Initialization


In [28]:
import numpy as np
from scipy.stats import norm

In [53]:
def bayesian_estimation(dist, elapsed_time, obs_var=0.001):
    prior_mean, prior_variance = dist.mean(), dist.var()


    updated_mean = (prior_mean / prior_variance + elapsed_time / obs_var) / (
        1 / prior_variance + 1 / obs_var
    )
    updated_variance = 1 / (1 / prior_variance + 1 / obs_var)

        
    dist = norm(loc=updated_mean, scale=max(updated_variance**0.5, 1e-3))
    

    print(f"Mean: ({prior_mean} -> {updated_mean})")
    return dist

In [65]:
from scipy.stats import norm
import numpy as np

# Configuration class for task settings
class Config:
    def __init__(self, criteria=0.7, interval=0.1, obs_dur=0.01):
        self.criteria = criteria
        self.interval = interval
        self.obs_dur = obs_dur

# Task information class
class TaskInfo:
    def __init__(self, idx, plan_task, sim_task, start_time):
        self.idx = idx
        self.plan_task = plan_task
        self.sim_task = sim_task
        self.start_time = start_time

# Bayesian Estimation function
def bayesian_estimation(dist, elapsed_time, obs_var=0.001):
    prior_mean, prior_variance = dist.mean(), dist.var()
    if prior_variance < 1e-6:
        prior_variance = 1e-6

    try:
        updated_mean = (prior_mean / prior_variance + elapsed_time / obs_var) / (
            1 / prior_variance + 1 / obs_var
        )
        updated_variance = 1 / (1 / prior_variance + 1 / obs_var)
        if not (np.isfinite(updated_mean) and np.isfinite(updated_variance)):
            updated_mean, updated_variance = prior_mean, prior_variance
            print("Warning: NaN encountered, using prior values.")
        dist = norm(loc=updated_mean, scale=max(updated_variance**0.5, 1e-3))
    except Exception as e:
        print(f"Exception during estimation: {e}, using prior values.")
        dist = norm(loc=prior_mean, scale=max(prior_variance**0.5, 1e-3))

    print(f"   [Estimation] Mean updated: {prior_mean:.2f} -> {updated_mean:.2f}")
    return dist

# Function to run a single task and estimate its duration
def run_task(task_info, config):
    print("\n===================================")
    print(f"Task {task_info.idx + 1}: {task_info.plan_task.name}")
    print("-----------------------------------")
    print(f"  - Ground Truth Start: {task_info.sim_task.start:.2f}")
    print(f"  - Ground Truth End: {task_info.sim_task.end:.2f}")
    print(f"  - Ground Truth Duration: {task_info.sim_task.duration:.2f}")
    print(f"  - Planned Start: {task_info.plan_task.start:.2f}")
    print(f"  - Planned End: {task_info.plan_task.end:.2f}")
    print(f"  - Initial Estimated Duration: {task_info.plan_task.duration:.2f}")
    print("-----------------------------------")

    task_duration_dist = norm(loc=task_info.plan_task.duration, scale=(task_info.plan_task.duration / 2))
    t_c = task_info.start_time

    while True:
        t_c += config.interval
        elapsed_time = t_c - task_info.start_time
        print(f"   [Time: {t_c:.2f}] Elapsed: {elapsed_time:.2f}", end='')

        if task_duration_dist.cdf(elapsed_time) >= config.criteria:
            task_duration_dist = bayesian_estimation(task_duration_dist, elapsed_time)

        if task_info.sim_task.end <= t_c:
            task_duration_dist = bayesian_estimation(task_duration_dist, elapsed_time)
            print("\n-----------------------------------")
            print(f"   [Task End] Actual End: {t_c:.2f}")
            print(f"   [Estimation] Final Estimated Duration: {task_duration_dist.mean():.2f}")
            print("===================================")
            break

    return t_c

# Main function to execute all tasks
def main(plan_tasks, sim_tasks):
    config = Config(criteria=0.7, interval=0.1, obs_dur=0.01)
    start_time = 0

    for idx, (plan_task, sim_task) in enumerate(zip(plan_tasks, sim_tasks)):
        task_info = TaskInfo(idx, plan_task, sim_task, start_time)
        start_time = run_task(task_info, config)




In [66]:
# Run the main function
main(plan_tasks, sim_tasks)


Task 1: Move (Living Room -> Kitchen:Toaster)
-----------------------------------
  - Ground Truth Start: 0.00
  - Ground Truth End: 1.25
  - Ground Truth Duration: 1.25
  - Planned Start: 0.00
  - Planned End: 0.75
  - Initial Estimated Duration: 0.75
-----------------------------------
   [Time: 0.10] Elapsed: 0.10   [Time: 0.20] Elapsed: 0.20   [Time: 0.30] Elapsed: 0.30   [Time: 0.40] Elapsed: 0.40   [Time: 0.50] Elapsed: 0.50   [Time: 0.60] Elapsed: 0.60   [Time: 0.70] Elapsed: 0.70   [Time: 0.80] Elapsed: 0.80   [Time: 0.90] Elapsed: 0.90   [Time: 1.00] Elapsed: 1.00   [Estimation] Mean updated: 0.75 -> 1.00
   [Time: 1.10] Elapsed: 1.10   [Estimation] Mean updated: 1.00 -> 1.05
   [Time: 1.20] Elapsed: 1.20   [Estimation] Mean updated: 1.05 -> 1.10
   [Time: 1.30] Elapsed: 1.30   [Estimation] Mean updated: 1.10 -> 1.15
   [Estimation] Mean updated: 1.15 -> 1.18

-----------------------------------
   [Task End] Actual End: 1.30
   [Estimation] Final Estimated Duration: 1.18

Ta